# 🎙️ Train Chatterbox TTS for Odia Voice Cloning (MIRA Engine)
**Hardware Target:** Google Colab Tesla T4 (15GB VRAM) or A100 (40GB/80GB)

This notebook fine-tunes Chatterbox's Audio Language Model (T3) on 17,809 native Odia clips.

### Quick Setup Steps:
1. In your **Google Drive**, create a folder named `chatterbox_odia`.
2. Upload your precomputed `cache_t3.pt` (104 MB) into that `chatterbox_odia/` folder.
3. Select Colab GPU Runtime: `Runtime` -> `Change runtime type` -> **T4 GPU**.
4. Click `Runtime` -> `Run all`.
5. All checkpoints and the final trained weights (`t3_odia_final.safetensors`) will automatically save straight back into your Google Drive.

In [ ]:
# --- Cell 1: Hardware & GPU Verification ---
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16 = torch.cuda.is_bf16_supported()
    print(f"Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"Precision Mode: {'Native BF16' if bf16 else 'Hardware FP16 + GradScaler (T4 Optimized)'}")
else:
    raise RuntimeError("GPU not detected! Please go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# --- Cell 2: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Cell 3: Synchronize PyTorch + Torchaudio & Install Dependencies ---
# 1. Reinstall synchronized PyTorch 2.6.0 + Torchaudio 2.6.0 for CUDA 12.4
!pip install --upgrade --force-reinstall torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
# 2. Install dependencies with NumPy < 2.4 (required by Numba/Librosa)
!pip install -q "numpy<2.4" chatterbox-tts bitsandbytes safetensors soundfile librosa

import torch
import torchaudio
import numpy as np
print(f"PyTorch: {torch.__version__} | Torchaudio: {torchaudio.__version__} | NumPy: {np.__version__} | CUDA: {torch.cuda.is_available()}")

from chatterbox.tts import ChatterboxTTS
print("✅ Chatterbox TTS and torchaudio loaded with 100% C++ symbol alignment!")

In [ ]:
# --- Cell 4: Verify Cache in Google Drive ---
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/chatterbox_odia')
CACHE_FILE = DRIVE_DIR / 'cache_t3.pt'
CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not CACHE_FILE.exists():
    print(f"❌ Cache file NOT found at: {CACHE_FILE}")
    print("👉 Make sure you uploaded 'cache_t3.pt' (104 MB) into 'chatterbox_odia' on your Google Drive.")
else:
    size_mb = CACHE_FILE.stat().st_size / (1024 * 1024)
    print(f"✅ Successfully located cache_t3.pt ({size_mb:.1f} MB) in Google Drive!")
    print(f"✅ Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
# --- Cell 5: Train Chatterbox Odia T3 ALM ---
import os, sys, glob, ctypes, time, logging
# Pre-load libcudart.so.12 into global symbol table
cuda12_libs = sorted(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib/libcudart.so*') +
                     glob.glob('/usr/local/cuda*/lib64/libcudart.so.12*'))
for lib in cuda12_libs:
    try:
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
    except Exception:
        pass
if cuda12_libs:
    os.environ['LD_LIBRARY_PATH'] = os.path.dirname(cuda12_libs[0]) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

from dataclasses import dataclass
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from chatterbox.tts import ChatterboxTTS
from chatterbox.models.t3.t3 import T3, T3Cond

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('colab_train')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
logger.info(f"Training on {torch.cuda.get_device_name(0)} with {'BF16' if use_bf16 else 'FP16 + GradScaler'}")

# 1. Vocab Resizing Helper
def resize_and_load_t3_weights(new_model, pretrained_state_dict):
    new_state = new_model.state_dict()
    emb_name = 'text_emb.weight'
    head_name = 'text_head.weight'
    for name, param in pretrained_state_dict.items():
        if name in (emb_name, head_name):
            continue
        if name in new_state and new_state[name].shape == param.shape:
            new_state[name].copy_(param)
    for name in (emb_name, head_name):
        old_w = pretrained_state_dict[name]
        old_n = old_w.shape[0]
        new_n = new_state[name].shape[0]
        new_state[name][:old_n, :].copy_(old_w)
        if new_n > old_n:
            mean_row = old_w.mean(dim=0)
            new_state[name][old_n:, :].copy_(mean_row.unsqueeze(0).expand(new_n - old_n, -1))
        logger.info(f"{name}: {old_n} -> {new_n} tokens ({new_n - old_n} mean-initialized for Odia)")
    new_model.load_state_dict(new_state)
    return new_model

# 2. Collator
@dataclass
class SpeechDataCollator:
    t3_config: any
    text_pad_token_id: int
    speech_pad_token_id: int
    def __call__(self, features):
        features = [f for f in features if f is not None]
        if not features:
            return {}
        batch_size = len(features)
        text_tokens_list = [f['text_tokens'] for f in features]
        speech_tokens_list = [f['speech_tokens'] for f in features]
        max_text_len = max(len(t) for t in text_tokens_list)
        max_speech_len = max(len(t) for t in speech_tokens_list)
        padded_text_tokens = torch.stack([
            F.pad(t, (0, max_text_len - len(t)), value=self.text_pad_token_id) for t in text_tokens_list
        ])
        padded_speech_tokens = torch.stack([
            F.pad(s, (0, max_speech_len - len(s)), value=self.speech_pad_token_id) for s in speech_tokens_list
        ])
        text_token_lens = torch.stack([f['text_token_lens'] for f in features])
        speech_token_lens = torch.stack([f['speech_token_lens'] for f in features])
        t3_cond_speaker_emb = torch.stack([f['t3_cond_speaker_emb'] for f in features])
        t3_cond_prompt_speech_tokens = torch.stack([f['t3_cond_prompt_speech_tokens'] for f in features])
        emotion_adv_scalars = torch.stack([f['t3_cond_emotion_adv'] for f in features])
        t3_cond_emotion_adv = emotion_adv_scalars.view(batch_size, 1, 1)
        return {
            'text_tokens': padded_text_tokens,
            'text_token_lens': text_token_lens,
            'speech_tokens': padded_speech_tokens,
            'speech_token_lens': speech_token_lens,
            't3_cond_speaker_emb': t3_cond_speaker_emb,
            't3_cond_prompt_speech_tokens': t3_cond_prompt_speech_tokens,
            't3_cond_emotion_adv': t3_cond_emotion_adv,
        }

# 3. Model Wrapper
class T3ForFineTuning(torch.nn.Module):
    def __init__(self, t3_model: T3, t3_config):
        super().__init__()
        self.t3 = t3_model
        self.t3_config = t3_config
    def forward(self, text_tokens, text_token_lens, speech_tokens, speech_token_lens,
                t3_cond_speaker_emb, t3_cond_prompt_speech_tokens, t3_cond_emotion_adv):
        cond = T3Cond(
            speaker_emb=t3_cond_speaker_emb,
            cond_prompt_speech_tokens=t3_cond_prompt_speech_tokens,
            cond_prompt_speech_emb=None,
            emotion_adv=t3_cond_emotion_adv
        ).to(device=self.t3.device)
        out = self.t3.forward(
            t3_cond=cond,
            text_tokens=text_tokens,
            text_token_lens=text_token_lens,
            speech_tokens=speech_tokens,
            speech_token_lens=speech_token_lens,
            training=True,
        )
        IGNORE_ID = -100
        dev = out.text_logits.device
        mask_text = torch.arange(text_tokens.size(1), device=dev)[None] >= text_token_lens[:, None]
        mask_speech = torch.arange(speech_tokens.size(1), device=dev)[None] >= speech_token_lens[:, None]
        masked_text = text_tokens.masked_fill(mask_text, IGNORE_ID)
        masked_speech = speech_tokens.masked_fill(mask_speech, IGNORE_ID)
        loss_text = F.cross_entropy(out.text_logits.transpose(1, 2), masked_text, ignore_index=IGNORE_ID)
        loss_speech = F.cross_entropy(out.speech_logits.transpose(1, 2), masked_speech, ignore_index=IGNORE_ID)
        return loss_text + loss_speech

# 4. Initialize Base Model
logger.info('Downloading Chatterbox base weights from HuggingFace (ResembleAI/chatterbox)...')
chatterbox_model = ChatterboxTTS.from_pretrained(device='cpu')
t3_model = chatterbox_model.t3
t3_config = t3_model.hp

NEW_VOCAB_SIZE = 781
if t3_config.text_tokens_dict_size != NEW_VOCAB_SIZE:
    old_state = t3_model.state_dict()
    t3_config.text_tokens_dict_size = NEW_VOCAB_SIZE
    t3_model = T3(hp=t3_config)
    t3_model = resize_and_load_t3_weights(t3_model, old_state)

for p in t3_model.parameters():
    p.requires_grad = True
t3_model = t3_model.to(device)
t3_model.train()
wrapped = T3ForFineTuning(t3_model, t3_config).to(device)

# 5. Load Precomputed Cache
logger.info(f'Loading {CACHE_FILE}...')
cached_items = torch.load(str(CACHE_FILE), weights_only=False)
logger.info(f'Loaded {len(cached_items)} clips into RAM!')

collator = SpeechDataCollator(t3_config, t3_config.stop_text_token, t3_config.stop_speech_token)
BATCH_SIZE = 2
GRAD_ACCUM = 8
loader = DataLoader(cached_items, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, num_workers=2, pin_memory=True)

# 6. Optimizer & Scaler
try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.AdamW8bit(t3_model.parameters(), lr=2e-5)
    logger.info('Using 8-bit AdamW (bitsandbytes) for maximum VRAM efficiency.')
except ImportError:
    optimizer = torch.optim.AdamW(t3_model.parameters(), lr=2e-5)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda' and not use_bf16))

# 7. Resume if Checkpoint Exists
step = 0
from safetensors.torch import load_file, save_file
existing_ckpts = sorted(CHECKPOINT_DIR.glob('t3_odia_step*.safetensors'), key=lambda p: int(p.stem.rsplit('step', 1)[1]) if 'step' in p.stem else 0)
if existing_ckpts:
    latest = existing_ckpts[-1]
    t3_model.load_state_dict(load_file(str(latest)))
    step = int(latest.stem.rsplit('step', 1)[1])
    logger.info(f'Resumed from checkpoint: {latest} (Step {step})')

# 8. Training Loop
TOTAL_STEPS = len(loader)
t0 = time.time()
logger.info(f'Starting training: {TOTAL_STEPS} steps, batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM} (Effective Batch={BATCH_SIZE*GRAD_ACCUM})')

for batch_idx, batch in enumerate(loader):
    if not batch:
        continue
    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type == 'cuda')):
        loss = wrapped(**batch)
        loss_scaled = loss / GRAD_ACCUM
    if scaler.is_enabled():
        scaler.scale(loss_scaled).backward()
    else:
        loss_scaled.backward()
    step += 1
    if step % GRAD_ACCUM == 0:
        if scaler.is_enabled():
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(t3_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(t3_model.parameters(), max_norm=1.0)
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    if step % 10 == 0 or step == 1:
        elapsed = time.time() - t0
        speed = step / max(elapsed, 1e-4)
        eta_h = (TOTAL_STEPS - step) / max(speed, 1e-4) / 3600
        print(f'Step {step}/{TOTAL_STEPS} | Loss: {loss.item():.4f} | Speed: {1.0/speed:.2f}s/step | ETA: {eta_h:.1f}h')
    if step % 250 == 0:
        ckpt_path = CHECKPOINT_DIR / f't3_odia_step{step}.safetensors'
        save_file(t3_model.state_dict(), str(ckpt_path))
        print(f'💾 Saved checkpoint to Drive: {ckpt_path}')

final_path = CHECKPOINT_DIR / 't3_odia_final.safetensors'
save_file(t3_model.state_dict(), str(final_path))
print(f'🎉 Training complete! Final weights saved: {final_path}')

In [ ]:
# --- Cell 6: Test Odia Voice Generation Live in Colab ---
from IPython.display import Audio, display
import soundfile as sf

print('Synthesizing test speech in Odia using newly fine-tuned weights...')
test_text = 'ନମସ୍କାର! ମୁଁ ମୀରା, ଆପଣଙ୍କ ଓଡ଼ିଆ ଭଏସ୍ ଆସିଷ୍ଟାଣ୍ଟ। ଆଜି ଆପଣ କଣ ଅର୍ଡର କରିବାକୁ ଚାହାଁନ୍ତି?'

chatterbox_model.t3 = t3_model.eval()
wav = chatterbox_model.generate(test_text)

test_out = '/content/odia_test_sample.wav'
sf.write(test_out, wav, chatterbox_model.sr)
print(f'Audio saved to {test_out}')
display(Audio(test_out, autoplay=True))